# Lab 11 · PydanticAI: typed output is only the first check
## Goal
Build a typed equipment quote with dependency injection, a real tool round-trip, an output validator, and bounded requests. Test it using PydanticAI's `FunctionModel`, then optionally replace that model with Gemini. Prerequisites: Lab 5 and textbook chapters 5, 11, and 12; Python dataclasses and Pydantic models.

Estimated 110 minutes: 15 contracts, 20 worked example, 40 implementation, 20 failure cases, 15 report. You will demonstrate a schema-valid but factually invalid quote and reject it using the supplied catalog. This is a proposal only: no inventory is reserved and no payment is made.

## Setup
Use the separate `labs/frameworks/requirements.txt` environment described in this track's README. Do not upgrade the earlier lab environment in place. Each notebook is self-contained after installation. Python 3.12 is the tested interpreter family.

Default execution uses invented equipment-service fixtures and real framework runtimes, without network inference. These checks establish application behavior, not model accuracy. Student tasks are disabled until you implement them and set `RUN_EXERCISES=True`. Instructor copies enable them. Optional live cells require explicit `RUN_LIVE=True`, `GEMINI_API_KEY`, and `GEMINI_MODEL`. Choose a model available to your account; API use consumes quota. Keep credentials out of notebook cells and saved output.

In [1]:
import os
from importlib.metadata import version
RUN_EXERCISES = False
RUN_LIVE = False

def live_config():
    key, model = os.getenv("GEMINI_API_KEY"), os.getenv("GEMINI_MODEL")
    if not key or not model:
        raise RuntimeError("Set GEMINI_API_KEY and GEMINI_MODEL before enabling live calls")
    return key, model

print({p: version(p) for p in ("langgraph", "langchain", "pydantic-ai-slim")})

{'langgraph': '1.2.14', 'langchain': '1.4.3', 'pydantic-ai-slim': '2.54.0'}


## Steps
### 1. Define a typed output and request-scoped data
The catalog is trusted application data in this lab. Prices are integer cents to avoid binary floating-point currency arithmetic. A valid JSON shape cannot establish that the price equals the current catalog. The validator must consult the injected dependency. The test model will first call `catalog_price`, then emit a typed output tool call.

In [2]:
from dataclasses import dataclass
from pydantic import BaseModel, ConfigDict, Field, ValidationError
import pydantic_ai
pydantic_ai.BANNER_ENABLED = False
from pydantic_ai import Agent, RunContext, ModelRetry
from pydantic_ai.models.function import FunctionModel
from pydantic_ai.messages import ModelResponse, ToolCallPart, ToolReturnPart
from pydantic_ai.usage import UsageLimits
from pydantic_ai.exceptions import UnexpectedModelBehavior

class Quote(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    item: str = Field(min_length=1)
    quantity: int = Field(ge=1, le=5)
    total_cents: int = Field(ge=0, le=100_000)

@dataclass(frozen=True)
class Catalog:
    prices: dict[str, int]

catalog = Catalog(prices={"camera":2500, "tripod":500})

def scripted_quote(messages, info):
    # Deterministic protocol fixture: fixed task, not natural-language understanding.
    returns = [part for message in messages for part in message.parts if isinstance(part, ToolReturnPart) and part.tool_name == "catalog_price"]
    if not returns:
        return ModelResponse(parts=[ToolCallPart("catalog_price", {"item":"camera"}, tool_call_id="price-1")])
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {"item":"camera", "quantity":2, "total_cents":returns[-1].content * 2}, tool_call_id="quote-1")])

agent = Agent(FunctionModel(scripted_quote), deps_type=Catalog, output_type=Quote, retries=1)

@agent.tool
def catalog_price(ctx: RunContext[Catalog], item: str) -> int:
    """Read the unit price in cents for a catalog item."""
    if item not in ctx.deps.prices:
        raise ModelRetry("Choose an item in the supplied catalog")
    return ctx.deps.prices[item]

@agent.output_validator
def verify_quote(ctx: RunContext[Catalog], output: Quote) -> Quote:
    price = ctx.deps.prices.get(output.item)
    if price is None or output.total_cents != price * output.quantity:
        raise ModelRetry("Quote must agree with the supplied catalog")
    return output

result = await agent.run("Quote two cameras", deps=catalog, usage_limits=UsageLimits(request_limit=3))
assert result.output == Quote(item="camera",quantity=2,total_cents=5000)
print(result.output.model_dump())

{'item': 'camera', 'quantity': 2, 'total_cents': 5000}


### 2. Make the semantic failure visible
The following output passes the shape constraints but contains the wrong price. Schema validation and factual validation are separate operations. The model can receive retry feedback, but a persistent error must terminate. Never catch that termination and replace the output with an invented successful quote.

In [3]:
wrong = Quote(item="camera", quantity=2, total_cents=1)
assert wrong.total_cents != catalog.prices[wrong.item] * wrong.quantity

def always_wrong(messages, info):
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, wrong.model_dump(), tool_call_id="wrong-quote")])

with agent.override(model=FunctionModel(always_wrong)):
    try:
        await agent.run("Quote two cameras", deps=catalog, usage_limits=UsageLimits(request_limit=3))
    except UnexpectedModelBehavior:
        print("Persistent invalid price rejected after bounded retries.")
    else:
        raise AssertionError("invalid quote escaped semantic validation")

for bad in (0, 6, True, "2"):
    try: Quote(item="camera",quantity=bad,total_cents=5000)
    except ValidationError: pass
    else: raise AssertionError("invalid quantity accepted")

Persistent invalid price rejected after bounded retries.


### 3. TODO: validate dependencies before inference (35 points)
Implement `checked_catalog(prices)` returning `Catalog` with a defensive dictionary copy. Accept an empty catalog. Reject non-dictionaries, empty or whitespace-only names, non-string names, booleans, non-integer cents, and prices outside 0–20,000. Raise `ValueError`. Why is `frozen=True` on a dataclass insufficient to freeze a contained dictionary?

In [4]:
def checked_catalog(prices):
    raise NotImplementedError("TODO: validate and copy dependency data")

In [5]:
if RUN_EXERCISES:
    assert checked_catalog({}).prices == {}
    source = {"camera":0, "tripod":20_000}
    copy_catalog = checked_catalog(source)
    source["camera"] = 5
    assert copy_catalog.prices["camera"] == 0
    for bad in (None,[],{"":1},{"x":True},{"x":-1},{"x":20_001},{"x":float("nan")},{1:100}):
        try: checked_catalog(bad)
        except ValueError: pass
        else: raise AssertionError("bad dependency accepted")

### 4. TODO: construct an independently testable typed agent (35 points)
Implement `make_quote_agent(model)` with the same typed dependency/output contract and bounded validation retries. Register a catalog lookup tool and a validator that enforces the item and total. Do not hard-code the fixture price; an injected catalog can change between runs. Match the tool name `catalog_price` expected by the fixture.

In [6]:
def make_quote_agent(model):
    raise NotImplementedError("TODO: create agent, register tool, register output validator")

In [7]:
if RUN_EXERCISES:
    app = make_quote_agent(FunctionModel(scripted_quote))
    for price in (0, 1200, 20_000):
        answer = await app.run("Quote two cameras", deps=checked_catalog({"camera":price}), usage_limits=UsageLimits(request_limit=3))
        assert answer.output.total_cents == price * 2
    bad_app = make_quote_agent(FunctionModel(always_wrong))
    try: await bad_app.run("Quote", deps=checked_catalog({}), usage_limits=UsageLimits(request_limit=3))
    except UnexpectedModelBehavior: pass
    else: raise AssertionError("unknown item accepted")
    print("Student typed-agent checks passed")

### 5. Optional Gemini integration
Use the same tool and validator with the Google model adapter. A valid result must still satisfy the catalog rule. The request cap is per run; it is not a monetary budget. Keep live results separate from deterministic fixture results.

In [8]:
if RUN_LIVE:
    from pydantic_ai.models.google import GoogleModel
    from pydantic_ai.providers.google import GoogleProvider
    key, model_name = live_config()
    provider_model = GoogleModel(model_name, provider=GoogleProvider(api_key=key))
    with agent.override(model=provider_model):
        live_result = await agent.run("Use catalog_price to quote exactly two cameras.", deps=catalog, usage_limits=UsageLimits(request_limit=3))
    print(live_result.output.model_dump())
else:
    print("Live Gemini request skipped.")

Live Gemini request skipped.


## Checks and submission
Submit the completed notebook and an explanation of schema validity, catalog consistency, and authorization. Add a test where the model requests an unknown item and exhausts retries; add an extra-field output test. Explain what would have to change if prices could change after the quote but before purchase.

Rubric: dependency validation 35; typed agent integration 35; added failure tests 15; semantic/authorization explanation 15. Neither the fixture nor its success proves that a live model will select the right item or quantity. A production contract should also validate the requested quantity and item against the user's confirmed request.

## Next Steps
Compare this agent with Lab 10 and Lab 7 using the same fixtures, budgets, and output checks. Identify one requirement each framework makes easier to express. Avoid treating API ergonomics as evidence of better model reasoning.

References checked 2026-10-06: [PydanticAI testing](https://pydantic.dev/docs/ai/guides/testing/), [agents](https://pydantic.dev/docs/ai/core-concepts/agent/), [output validation](https://pydantic.dev/docs/ai/core-concepts/output/), [Google models](https://pydantic.dev/docs/ai/models/google/). Use the pinned package implementation as the reproducibility baseline when mutable documentation changes.